# Análise do Desenvolvimento Humano e Econômico Global com Gapminder

Este trabalho analisa dados de países entre 1952 e 2007 para entender a relação entre expectativa de vida, renda, população e continente.

A ordem da apresentação é simples:

1. limpar e validar a base;
2. observar tendências gerais;
3. comparar continentes e países;
4. medir associações com correlação e regressão;
5. complementar com dados do Banco Mundial;
6. gerar um HTML final com os gráficos interativos.

A ideia principal não é provar causa e efeito. A ideia é mostrar padrões nos dados e explicar o que eles sugerem.

## 0. Setup e bibliotecas

As bibliotecas foram escolhidas de acordo com o papel de cada etapa da análise:

- `pandas`: leitura do CSV, limpeza, agrupamentos, rankings e tabelas. É a base do trabalho porque o dataset é tabular.
- `numpy`: cálculos numéricos dentro de `src/advanced_analysis.py`, especialmente `np.log` para criar variáveis em escala logarítmica e `np.average` para médias ponderadas por população. O notebook não importa `np` diretamente porque esses cálculos foram encapsulados em funções reutilizáveis.
- `plotly`: gráficos interativos e mapa mundial. Foi escolhido porque o resultado final em HTML permite explorar países e valores com hover.
- `scipy`: cálculo das correlações Pearson e Spearman. Ele transforma a leitura visual dos gráficos em medidas estatísticas.
- `statsmodels`: regressão exploratória com coeficiente, intervalo de confiança e R². É mais interpretável para análise estatística do que usar apenas uma biblioteca preditiva.
- `requests`: acesso à API do Banco Mundial, encapsulado no módulo `src/advanced_analysis.py`.
- `pathlib` e `importlib`: localização dos módulos próprios em `src` sem depender do diretório em que o VS Code ou o `nbconvert` iniciam o kernel.

O arquivo `src/advanced_analysis.py` concentra funções reutilizáveis para que o notebook fique mais limpo e organizado. Essa separação deixa a apresentação mais legível: o notebook mostra a linha de raciocínio e o módulo guarda detalhes técnicos repetíveis.

In [ ]:
_Path = __import__("pathlib").Path
_root = next(
    candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
    if (candidate / "src" / "notebook_state.py").exists()
)
exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

print("Setup concluído.")

## 1. Carregamento e limpeza

O arquivo original fica preservado em `data/raw/gapminder_full.csv`. A análise parte de uma cópia limpa, sem alterar o CSV bruto.

Código central:

```python
df_raw = pd.read_csv(DATA_RAW)
df_clean = df_raw.dropna().drop_duplicates().copy()
```

`read_csv` carrega a base como um `DataFrame`. Em seguida, `dropna` trata registros com valores ausentes, `drop_duplicates` remove linhas repetidas e `copy` cria uma base separada para as próximas etapas.

Essa limpeza é importante porque duplicatas ou ausências podem distorcer médias, rankings, correlações e regressões.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

df_raw = pd.read_csv(DATA_RAW)
df_clean = df_raw.dropna().drop_duplicates().copy()

limpeza = pd.DataFrame({
    "etapa": ["Base bruta", "Após dropna + drop_duplicates"],
    "linhas": [len(df_raw), len(df_clean)],
    "colunas": [df_raw.shape[1], df_clean.shape[1]],
    "ausentes_total": [df_raw.isna().sum().sum(), df_clean.isna().sum().sum()],
    "duplicatas": [df_raw.duplicated().sum(), df_clean.duplicated().sum()],
})
limpeza

### Leitura da limpeza

A base bruta tinha 1.736 linhas. Depois da limpeza, ficaram 1.704 observações, com 32 duplicatas removidas.

Esse resultado é coerente com a estrutura esperada do Gapminder:

```text
142 países x 12 anos = 1.704 observações
```

Depois da limpeza, cada país tem uma observação por ano analisado.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

estrutura = pd.DataFrame({
    "métrica": ["países", "anos", "continentes", "primeiro ano", "último ano"],
    "valor": [
        df_clean["country"].nunique(),
        df_clean["year"].nunique(),
        df_clean["continent"].nunique(),
        df_clean["year"].min(),
        df_clean["year"].max(),
    ],
})
estrutura

## 2. Validação formal da base

Depois da limpeza, validamos a base com a função:

```python
validate_gapminder_schema(df_clean)
```

Essa função verifica se:

- as colunas esperadas existem;
- não restam valores ausentes;
- não restam duplicatas;
- população, expectativa de vida e PIB per capita são positivos;
- os continentes pertencem ao conjunto esperado;
- os anos estão no intervalo analisado.

A ideia é simples: antes de interpretar gráficos, precisamos garantir que a base obedece às regras mínimas esperadas. Isso torna a análise mais reprodutível e defensável.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

validate_gapminder_schema(df_clean)

## 3. Tendências globais

Nesta etapa, os dados são agrupados por ano com `groupby('year')`.

Para cada ano, calculamos a expectativa de vida média, o PIB per capita médio e a população total coberta pela base.

Esse primeiro recorte mostra a tendência geral do período antes de entrar em comparações por continente ou por país.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

global_year = (
    df_clean.groupby("year")
    .agg(
        lifeExp_media=("lifeExp", "mean"),
        gdpPercap_medio=("gdpPercap", "mean"),
        pop_total=("pop", "sum"),
    )
    .reset_index()
)

fig_life_global = px.line(
    global_year, x="year", y="lifeExp_media", markers=True,
    title="Expectativa de vida média global (1952-2007)",
    labels={"year": "Ano", "lifeExp_media": "Expectativa de vida média"},
)
show_plot(fig_life_global)

### Interpretação

A expectativa de vida média global cresce de forma consistente. Essa leitura mostra progresso geral, mas ainda não revela desigualdades regionais. Por isso, a próxima etapa divide a análise por continente.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

if "global_year" not in globals():
    global_year = (
        df_clean.groupby("year")
        .agg(
            lifeExp_media=("lifeExp", "mean"),
            gdpPercap_medio=("gdpPercap", "mean"),
            pop_total=("pop", "sum"),
        )
        .reset_index()
    )

fig_gdp_global = px.line(
    global_year, x="year", y="gdpPercap_medio", markers=True,
    title="PIB per capita médio global (1952-2007)",
    labels={"year": "Ano", "gdpPercap_medio": "PIB per capita médio"},
)
show_plot(fig_gdp_global)

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

if "global_year" not in globals():
    global_year = (
        df_clean.groupby("year")
        .agg(
            lifeExp_media=("lifeExp", "mean"),
            gdpPercap_medio=("gdpPercap", "mean"),
            pop_total=("pop", "sum"),
        )
        .reset_index()
    )

fig_pop_global = px.area(
    global_year, x="year", y=global_year["pop_total"] / 1_000_000_000,
    title="População total representada no Gapminder (1952-2007)",
    labels={"year": "Ano", "y": "População (bilhões)"},
)
fig_pop_global.update_traces(mode="lines")
show_plot(fig_pop_global)

## 4. Comparação por continente

A média global resume o mundo inteiro, mas pode esconder diferenças regionais. Por isso, também agrupamos os dados por continente e ano:

```python
df_clean.groupby(['continent', 'year'])
```

Esse recorte mostra quais regiões avançaram mais, quais permaneceram atrás e se as diferenças diminuíram ou continuaram ao longo do tempo.

A leitura por continente é útil, mas ainda é uma simplificação: dentro de cada continente existem países com realidades bem diferentes.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

continent_year = (
    df_clean.groupby(["continent", "year"], as_index=False)
    .agg(
        lifeExp_media=("lifeExp", "mean"),
        gdpPercap_medio=("gdpPercap", "mean"),
        pop_total=("pop", "sum"),
    )
)

fig_cont_life = px.line(
    continent_year, x="year", y="lifeExp_media", color="continent", markers=True,
    title="Expectativa de vida média por continente",
    labels={"year": "Ano", "lifeExp_media": "Expectativa de vida", "continent": "Continente"},
)
show_plot(fig_cont_life)

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

if "continent_year" not in globals():
    continent_year = (
        df_clean.groupby(["continent", "year"], as_index=False)
        .agg(lifeExp_media=("lifeExp", "mean"), gdpPercap_medio=("gdpPercap", "mean"))
    )

fig_cont_gdp = px.line(
    continent_year, x="year", y="gdpPercap_medio", color="continent", markers=True,
    title="PIB per capita médio por continente",
    labels={"year": "Ano", "gdpPercap_medio": "PIB per capita", "continent": "Continente"},
)
show_plot(fig_cont_gdp)

### Interpretação

Europa e Oceania aparecem em patamares altos de renda e longevidade. A África melhora ao longo do período, mas termina em 2007 ainda com os menores níveis médios. Isso sustenta a conclusão central: houve progresso global, mas desigual.

## 5. Relação entre PIB per capita e expectativa de vida

Neste gráfico, cada ponto representa um país.

O eixo X mostra o PIB per capita, o eixo Y mostra a expectativa de vida, o tamanho da bolha representa a população e a cor indica o continente.

Usamos escala logarítmica no PIB porque a renda varia muito entre países. Essa escala evita que poucos países muito ricos comprimam visualmente os demais.

A leitura principal é que países com maior renda tendem a apresentar maior expectativa de vida. Ainda assim, o gráfico mostra associação, não causalidade.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

fig_gapminder = make_interactive_2007(df_clean)
show_plot(fig_gapminder)

### Interpretação

O gráfico mostra associação positiva entre renda e expectativa de vida: países com maior PIB per capita tendem a apresentar maior longevidade.

A forma da nuvem de pontos sugere uma relação não linear. Em níveis baixos de renda, aumentos no PIB per capita parecem associados a grandes ganhos de expectativa de vida. Em níveis altos, ganhos adicionais de renda estão associados a aumentos menores de longevidade.

Essa leitura é exploratória. O gráfico não prova que renda causa longevidade; ele mostra que as duas variáveis caminham juntas na base analisada.

## 6. Rankings de países em 2007

Os rankings tornam os padrões mais concretos. Em vez de falar apenas de médias, mostramos países específicos no ano mais recente da base.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

df_2007 = df_clean[df_clean["year"] == 2007].copy()

top_life = df_2007.nlargest(10, "lifeExp")[["country", "continent", "lifeExp", "gdpPercap"]]
bottom_life = df_2007.nsmallest(10, "lifeExp")[["country", "continent", "lifeExp", "gdpPercap"]]

pd.concat({"Maior expectativa de vida": top_life, "Menor expectativa de vida": bottom_life})

### Interpretação

Os países com maior expectativa de vida em 2007 são majoritariamente europeus e asiáticos de alta renda. Entre os menores valores, há forte concentração africana, com o Afeganistão aparecendo como exceção asiática no grupo inferior.

## 7. Crescimento entre 1952 e 2007

Aqui comparamos o primeiro e o último ano da base:

```python
df_clean[df_clean['year'].isin([1952, 2007])]
```

Essa comparação resume 55 anos de mudança. Calculamos dois tipos de ganho:

- ganho absoluto: diferença direta entre 2007 e 1952;
- ganho percentual: crescimento relativo em relação ao ponto de partida.

Essa distinção importa muito. Um continente pode crescer bastante em percentual porque partiu de uma base baixa, mas ainda assim continuar distante em valores absolutos dos continentes mais ricos.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

extremos = df_clean[df_clean["year"].isin([1952, 2007])]

growth_life = extremos.groupby(["continent", "year"])["lifeExp"].mean().unstack()
growth_life["ganho_anos"] = growth_life[2007] - growth_life[1952]
growth_gdp = extremos.groupby(["continent", "year"])["gdpPercap"].mean().unstack()
growth_gdp["ganho_usd"] = growth_gdp[2007] - growth_gdp[1952]
growth_gdp["ganho_percentual"] = (growth_gdp[2007] / growth_gdp[1952] - 1) * 100

display(growth_life.round(2).sort_values("ganho_anos", ascending=False))
display(growth_gdp.round(2).sort_values("ganho_usd", ascending=False))

### Interpretação

A comparação entre ganho absoluto e percentual é importante. Um crescimento percentual alto em uma base baixa pode não reduzir a distância absoluta em relação a regiões que já eram ricas.

# Parte complementar: estatística, dados externos e visualizações interativas

Esta seção aprofunda a análise principal. Ela adiciona estatística, dados externos do Banco Mundial, mapa mundial, médias ponderadas e comparação de países específicos.

A função desta parte é reforçar a interpretação, não substituir a análise exploratória inicial. As conclusões continuam sendo associações observadas nos dados, não afirmações causais definitivas.

## 8. Correlações: Pearson e Spearman

Correlação mede a força e a direção da associação entre duas variáveis. O valor geralmente varia de -1 a 1.

- Valores próximos de 1 indicam associação positiva forte.
- Valores próximos de 0 indicam pouca associação.
- Valores próximos de -1 indicam associação negativa forte.

### Pearson

Pearson mede associação linear. Ele é adequado quando queremos avaliar se duas variáveis se movem de forma aproximadamente proporcional.

No trabalho, Pearson ajuda a verificar numericamente a relação entre renda e expectativa de vida observada nos gráficos.

### Spearman

Spearman usa ranking. Ele não exige uma relação em formato de reta; basta que valores maiores de uma variável estejam associados, em geral, a valores maiores da outra.

As duas medidas ajudam a quantificar associação, mas nenhuma delas prova causalidade.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

correlacoes = correlation_table(df_clean)
correlacoes

### Interpretação

A associação entre `lifeExp` e `log_gdpPercap` ficou mais forte do que entre `lifeExp` e `gdpPercap` bruto.

Isso reforça a decisão de usar escala logarítmica para renda, porque a relação entre PIB per capita e expectativa de vida não cresce de forma perfeitamente linear na escala original.

Também aparece que população, sozinha, tem associação bem mais fraca. Um país ser muito populoso não significa automaticamente ter maior expectativa de vida ou maior renda média.

## 9. Regressão exploratória

Modelo usado:

```text
lifeExp ~ log(gdpPercap)
```

A regressão estima a relação entre uma variável resposta e uma ou mais variáveis explicativas. Neste caso, a variável resposta é `lifeExp`, e a variável explicativa é `log(gdpPercap)`.

A tabela deve ser lida assim:

- `coef_log_gdpPercap`: direção e intensidade média da relação entre renda e expectativa de vida.
- `ic95_min` e `ic95_max`: intervalo de confiança de 95% para o coeficiente.
- `r2`: proporção da variação de `lifeExp` explicada pelo modelo.
- `n`: quantidade de observações usadas.

A regressão permite modelar a relação entre renda e expectativa de vida e poderia apoiar previsões em outro contexto. Aqui, o uso é exploratório: o modelo ajuda a interpretar padrões, sem afirmar causalidade.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

regressoes = regression_summary(df_clean)
regressoes

### Interpretação

O coeficiente positivo de `log(gdpPercap)` indica que maior PIB per capita está associado, em média, a maior expectativa de vida.

O R² próximo de 0,65 indica que o modelo explica uma parte importante da variação da expectativa de vida, mas não toda a variação.

A parte não explicada pode estar relacionada a fatores que não entraram nesse modelo, como saneamento, educação, desigualdade, vacinação, conflitos, políticas públicas e acesso à saúde.

### Complemento avançado: regressão quantílica

A regressão linear estima a relação média entre renda e expectativa de vida. Como complemento, usamos regressão quantílica para observar essa relação em diferentes partes da distribuição.

Os quantis usados foram:

- 0,25: observações com menor expectativa de vida;
- 0,50: mediana;
- 0,75: observações com maior expectativa de vida.

Essa técnica ajuda a verificar se a associação entre renda e longevidade é parecida em todos os grupos ou se muda quando olhamos países em situações mais ou menos favoráveis.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

regressoes_quantilicas = quantile_regression_summary(df_clean)
regressoes_quantilicas

### Interpretação

Os coeficientes ficaram positivos nos três quantis. A mensagem principal se mantém: maior renda está associada a maior expectativa de vida.

O coeficiente maior no quantil 0,25 sugere que a renda aparece mais fortemente associada à expectativa de vida entre observações de menor longevidade.

O `pseudo_r2` é uma medida auxiliar da regressão quantílica e não deve ser comparado diretamente com o R² da regressão linear.

## 10. Dados externos do Banco Mundial

Para enriquecer a análise, adicionamos dois indicadores do Banco Mundial:

- Gini: medida de desigualdade de renda;
- mortalidade infantil: mortes de crianças antes de 1 ano a cada 1.000 nascidos vivos.

Esses indicadores complementam o Gapminder porque PIB per capita é uma média e não mostra distribuição de renda. Além disso, expectativa de vida é um indicador amplo, enquanto mortalidade infantil aproxima condições básicas de saúde, saneamento e cuidado público.

Com isso, a análise deixa de olhar apenas renda média e passa a considerar também desigualdade e condições sociais.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

world_bank = build_world_bank_dataset(WB_RAW, refresh=False)
enriched_2007 = enrich_gapminder_2007_with_world_bank(df_clean, world_bank, WB_ENRICHED_2007)

coverage = pd.DataFrame({
    "indicador": ["Gini mais recente até 2007", "Mortalidade infantil mais recente até 2007"],
    "países com dado": [
        enriched_2007["gini_index_latest"].notna().sum(),
        enriched_2007["infant_mortality_per_1000_latest"].notna().sum(),
    ],
    "total de países": [len(enriched_2007), len(enriched_2007)],
})
coverage["cobertura (%)"] = (coverage["países com dado"] / coverage["total de países"] * 100).round(1)
coverage

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

if "enriched_2007" not in globals():
    world_bank = build_world_bank_dataset(WB_RAW, refresh=False)
    enriched_2007 = enrich_gapminder_2007_with_world_bank(df_clean, world_bank, WB_ENRICHED_2007)

fig_wb = make_world_bank_scatter(enriched_2007)
show_plot(fig_wb)

### Interpretação

Gini e mortalidade infantil ajudam a ir além do PIB per capita. Dois países podem ter renda média parecida, mas distribuição de renda, saneamento, vacinação, saúde pública e mortalidade infantil muito diferentes.

A cobertura também precisa ser considerada: mortalidade infantil tem cobertura maior que Gini. Por isso, qualquer conclusão usando Gini deve ser apresentada com mais cautela.

## 11. Mapa mundial de expectativa de vida

Aqui usamos `plotly.express.choropleth`. Um choropleth é um mapa em que cada país recebe uma cor de acordo com o valor de uma variável.

Neste caso:

- país: identificado por código ISO;
- cor: expectativa de vida em 2007;
- hover: nome do país e indicadores principais.

Por que usar mapa:

- evidencia desigualdade espacial rapidamente;
- facilita comunicação visual;
- transforma uma tabela de 142 países em uma leitura geográfica direta.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

fig_map = make_life_expectancy_choropleth(df_clean, year=2007)
show_plot(fig_map)

## 12. Média simples vs. média ponderada pela população

As duas médias respondem perguntas diferentes.

Na média simples, cada país tem o mesmo peso. Brasil, China, Índia e Islândia contam como uma unidade cada. Essa média responde como o país médio evoluiu.

Na média ponderada pela população, países mais populosos têm mais peso no cálculo. China e Índia influenciam mais o resultado global do que países pequenos. Essa média aproxima melhor a experiência da pessoa média considerando o peso populacional.

A diferença entre as duas médias é metodológica, não erro de cálculo.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

media_ponderada = population_weighted_life_expectancy(df_clean)
display(media_ponderada)

fig_weighted = make_weighted_life_expectancy_chart(df_clean)
show_plot(fig_weighted)

## 13. Comparação de países específicos

Países escolhidos:

- Brasil;
- China;
- Índia;
- Estados Unidos;
- Japão.

Por que esses países:

- Brasil: referência próxima e regionalmente relevante;
- China e Índia: países muito populosos, com grande peso na média ponderada;
- Estados Unidos: economia avançada e renda alta;
- Japão: destaque em longevidade.

Essa comparação ajuda a transformar padrões globais em trajetórias concretas e fáceis de explicar.

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

paises_foco = selected_countries_timeseries(df_clean)
resumo_paises = (
    paises_foco[paises_foco["year"].isin([1952, 2007])]
    .pivot(index="country", columns="year", values=["lifeExp", "gdpPercap", "pop_milhoes"])
)
resumo_paises.round(2)

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

fig_countries_life = make_selected_countries_chart(df_clean, metric="lifeExp")
show_plot(fig_countries_life)

In [ ]:
if "df_clean" not in globals():
    _Path = __import__("pathlib").Path
    _root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "notebook_state.py").exists()
    )
    exec((_root / "src" / "notebook_state.py").read_text(encoding="utf-8"), globals())

fig_countries_gdp = make_selected_countries_chart(df_clean, metric="gdpPercap")
show_plot(fig_countries_gdp)

## 14. Conclusão

O trabalho mostra que o mundo melhorou em expectativa de vida e renda entre 1952 e 2007, mas essa melhora não foi igual para todos.

A base foi limpa e validada antes da análise. Depois, os gráficos mostraram crescimento global da expectativa de vida, diferenças persistentes entre continentes e forte associação entre renda e longevidade.

As correlações quantificaram essas associações. A regressão reforçou que o PIB per capita, em escala logarítmica, explica uma parte relevante da variação na expectativa de vida, mas não tudo.

Os dados do Banco Mundial adicionaram desigualdade de renda e mortalidade infantil, deixando claro que desenvolvimento humano não pode ser resumido apenas por PIB.

A leitura final é exploratória: os dados revelam padrões fortes e bem documentados, mas não provam causa e efeito por si só.

## 15. Geração do HTML final

A célula abaixo existe apenas no notebook. Para apresentar, basta executar o play dessa última célula: ela gera `reports/analise_gapminder.html` e abre o arquivo no navegador.

Ela funciona em dois cenários:

- se o notebook já foi executado desde o começo, ela reutiliza a função preparada no setup;
- se o kernel foi reiniciado e você rodar só a última célula, ela localiza `src/report_builder.py` automaticamente.

O gerador cria uma cópia temporária do notebook sem as células marcadas como `remove_cell`. Isso evita que a célula final seja executada dentro da própria exportação e garante que o HTML final contenha apenas a análise, os textos e os gráficos.

In [ ]:
_build_html = globals().get("build_final_html")

if _build_html is None:
    _Path = __import__("pathlib").Path
    _importlib_util = __import__("importlib.util").util
    try:
        _display_module = __import__("IPython.display", fromlist=["HTML", "display"])
        _HTML = _display_module.HTML
        _display = _display_module.display
    except ModuleNotFoundError:
        _HTML = None
        _display = None

    _project_root = next(
        candidate for candidate in [_Path.cwd(), *_Path.cwd().parents]
        if (candidate / "src" / "report_builder.py").exists()
    )

    def _load_local_module(module_name, module_path):
        spec = _importlib_util.spec_from_file_location(module_name, module_path)
        if spec is None or spec.loader is None:
            raise ImportError(f"Não foi possível carregar {module_name} em {module_path}")
        module = _importlib_util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module

    _report_builder = _load_local_module(
        "report_builder",
        _project_root / "src" / "report_builder.py",
    )

    def _build_html(open_browser=True):
        print("Gerando HTML final da apresentação...")
        caminho = _report_builder.build_all_reports(open_browser=open_browser)
        if _HTML is not None and _display is not None:
            _display(_HTML(f"""
            <p><strong>HTML final gerado com sucesso.</strong></p>
            <p><a href="{caminho.as_uri()}" target="_blank">Abrir apresentação em HTML</a></p>
            <p><code>{caminho}</code></p>
            """))
        else:
            print(f"HTML final gerado com sucesso: {caminho}")
        return caminho

_build_html(open_browser=True)